# 02. Data Preprocessing & Canonical Feature Engineering
**Phase 2: Data Preprocessing**  
**Project:** Hospital Patient Data Analysis and Dashboard Development Using Python and Power BI

---

### Objective & Cleaning Scope
This notebook turns the raw dataset (`data/raw/healthcare_dataset.csv`) into an analysis-ready canonical dataset (`data/processed/hospital_clean.csv`).
Every cleaning decision is audited, executed, and justified:
1. **Audit Before Fixing**: Missing values, exact duplicates, near-duplicates, inconsistent text casing, date validation, age plausibility, negative billing, and categorical sets.
2. **Deterministic Cleansing**: Exact duplicate drop, near-duplicate flagging, date parsing, and text trimming.
3. **HIPAA-Compliant De-identification**: Salted SHA-256 patient pseudonymization (`patient_key`) and deterministic doctor pseudonymization (`doctor_id`). Original `Name` and `Doctor` permanently removed.
4. **Canonical & Temporal Feature Derivations**: `los_days`, `age_group`, `admission_month`, `admission_year`, `admission_quarter`, `admission_weekday`, `discharge_month`, `cost_per_day`, `high_cost_flag`, `long_stay_flag`, and 30-day readmission proxy logic.
5. **Validation Assertions**: Complete assertion suite validating schema, chronology, categories, and row-count reconciliation.


In [1]:
# Environment setup, imports, and seeds
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed
np.random.seed(42)

# Display configurations
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(f"Python interpreter: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")


C:\Users\berab\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\berab\AppData\Local\Temp\matplotlib-91r7xp8r because there was an issue with the default path (C:\Users\berab\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Python interpreter: 3.11.9
Pandas version: 2.2.1


## 1. Raw Dataset Audit Prior to Cleansing
Auditing data quality anomalies directly from `data/raw/healthcare_dataset.csv`.

In [2]:
raw_path = 'data/raw/healthcare_dataset.csv'
if not os.path.exists(raw_path):
    raw_path = '../data/raw/healthcare_dataset.csv'

df_raw = pd.read_csv(raw_path)
raw_rows, raw_cols = df_raw.shape
print(f"Raw Dataset Dimensions: {raw_rows:,} rows, {raw_cols} columns")

# Audit 1: Missing Values
raw_nulls = df_raw.isnull().sum()
print(f"Total Missing Values: {raw_nulls.sum()}")

# Audit 2: Exact Duplicates
exact_dups = df_raw.duplicated().sum()
print(f"Exact Duplicate Rows Detected: {exact_dups:,} ({exact_dups/raw_rows*100:.2f}%)")

# Audit 3: Date Parsing & Chronology
adm_parsed = pd.to_datetime(df_raw['Date of Admission'], format='%Y-%m-%d', errors='coerce')
dis_parsed = pd.to_datetime(df_raw['Discharge Date'], format='%Y-%m-%d', errors='coerce')
chrono_inversions = (dis_parsed < adm_parsed).sum()
print(f"Unparseable Dates: Admission={adm_parsed.isnull().sum()}, Discharge={dis_parsed.isnull().sum()}")
print(f"Chronological Inversions (Discharge < Admission): {chrono_inversions}")

# Audit 4: Age Plausibility
age_implausible = ((df_raw['Age'] < 0) | (df_raw['Age'] > 120)).sum()
print(f"Implausible Ages (outside 0-120): {age_implausible} (Min: {df_raw['Age'].min()}, Max: {df_raw['Age'].max()})")

# Audit 5: Billing Anomalies
neg_billing = (df_raw['Billing Amount'] < 0).sum()
zero_billing = (df_raw['Billing Amount'] == 0).sum()
print(f"Negative Billing Records: {neg_billing} (Min: ${df_raw['Billing Amount'].min():,.2f})")
print(f"Zero Billing Records: {zero_billing}")


Raw Dataset Dimensions: 55,500 rows, 15 columns
Total Missing Values: 0
Exact Duplicate Rows Detected: 534 (0.96%)
Unparseable Dates: Admission=0, Discharge=0
Chronological Inversions (Discharge < Admission): 0
Implausible Ages (outside 0-120): 0 (Min: 13, Max: 89)
Negative Billing Records: 108 (Min: $-2,008.49)
Zero Billing Records: 0


## 2. Execute Data Cleaning Pipeline via `src.clean`
Executing modular `clean_dataset(raw_df)` function, producing canonical dataset and quality audit log.

In [3]:
# Add src to sys.path if not present
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from src.clean import clean_dataset

clean_df, quality_log_df = clean_dataset(df_raw)
print(f"Cleaned Canonical Dataset Dimensions: {clean_df.shape[0]:,} rows, {clean_df.shape[1]} columns")
print("\n--- Data Quality Action Log ---")
print(quality_log_df[['issue', 'rows_affected', 'action']].to_string(index=False))


Cleaned Canonical Dataset Dimensions: 54,966 rows, 32 columns

--- Data Quality Action Log ---
                                                                    issue  rows_affected                                                                                   action
                                 Missing / Null values across all columns              0                                                          Retained (No imputation needed)
                           Exact duplicate rows across all source columns            534                                                         Dropped 534 exact duplicate rows
            Inconsistent whitespace and mixed casing in string attributes          55500                         Trimmed leading/trailing spaces and standardized text formatting
                               Direct patient identifier in 'Name' column          55500                 Generated salted SHA-256 12-char pseudonym 'patient_key'; dropped 'Name'
               

## 3. Canonical Schema & Derived Feature Verification
Inspecting all engineered features, clinical flags, and proxy metrics.

In [4]:
# Summarize canonical derived metrics
print("Canonical Columns List:")
print(clean_df.columns.tolist())

# Summary of Flags and Derived Indicators
flag_summary = pd.DataFrame({
    'Metric / Indicator': [
        'Total Admissions (Clean)',
        'Unique Patients (patient_key)',
        'Unique Doctors (doctor_id)',
        'Near-Duplicate Admissions (near_duplicate_flag=1)',
        'Financial Outlier Stays (high_cost_flag=1)',
        'Extended Stay Admissions (long_stay_flag=1)',
        'Billing Outlier Admissions (billing_outlier_flag=1)',
        '30-Day Readmissions (readmitted_30d=1)',
        '30-Day Readmission Eligible Stays',
        '30-Day Readmission Rate (%)'
    ],
    'Value': [
        f"{len(clean_df):,}",
        f"{clean_df['patient_key'].nunique():,}",
        f"{clean_df['doctor_id'].nunique():,}",
        f"{clean_df['near_duplicate_flag'].sum():,}",
        f"{clean_df['high_cost_flag'].sum():,}",
        f"{clean_df['long_stay_flag'].sum():,}",
        f"{clean_df['billing_outlier_flag'].sum():,}",
        f"{clean_df['readmitted_30d'].sum():,}",
        f"{clean_df['readmission_eligible'].sum():,}",
        f"{(clean_df[clean_df['readmission_eligible'] == 1]['readmitted_30d'].mean() * 100):.2f}%"
    ]
})
print("\nCanonical Metric & Derived Feature Summary:")
print(flag_summary.to_string(index=False))


Canonical Columns List:
['admission_id', 'patient_key', 'age', 'age_group', 'gender', 'blood_type', 'medical_condition', 'admission_date', 'discharge_date', 'los_days', 'admission_month', 'admission_year', 'admission_quarter', 'admission_weekday', 'discharge_month', 'doctor_id', 'hospital', 'insurance_provider', 'billing_amount', 'cost_per_day', 'room_number', 'admission_type', 'medication', 'test_results', 'high_cost_flag', 'long_stay_flag', 'days_to_next_admission', 'readmitted_30d', 'readmission_eligible', 'near_duplicate_flag', 'billing_outlier_flag', 'los_outlier_flag']

Canonical Metric & Derived Feature Summary:
                                 Metric / Indicator  Value
                           Total Admissions (Clean) 54,966
                      Unique Patients (patient_key) 40,235
                         Unique Doctors (doctor_id) 40,341
  Near-Duplicate Admissions (near_duplicate_flag=1)  9,938
         Financial Outlier Stays (high_cost_flag=1)  2,749
        Extended St

## 4. Before-and-After Cleaning Comparison Table
Systematic reconciliation of dimensions, nulls, duplicates, and column counts.

In [5]:
comparison_table = pd.DataFrame({
    'Dimension / Attribute': [
        'Total Rows',
        'Exact Duplicate Rows',
        'Direct Identifiers Present',
        'Total Missing Values (Nulls)',
        'Total Feature Columns',
        'Date Data Types',
        'Negative Billed Records'
    ],
    'Before (Raw Data)': [
        f"{raw_rows:,}",
        f"{exact_dups:,}",
        "Yes (Name, Doctor)",
        "0 (100% Complete)",
        f"{raw_cols}",
        "Object (String)",
        f"{neg_billing:,}"
    ],
    'After (Clean Data)': [
        f"{len(clean_df):,}",
        "0 (All Dropped)",
        "No (Pseudonymized & Dropped)",
        "0 (100% Complete)",
        f"{clean_df.shape[1]}",
        "Datetime64[ns]",
        f"{(clean_df['billing_amount'] < 0).sum():,} (Flagged)"
    ],
    'Delta / Transformation Action': [
        f"-{exact_dups:,} exact duplicate rows removed",
        "Purged identical repeats (First occurrence kept)",
        "patient_key (SHA-256) & doctor_id (DR-xxx)",
        "Maintained full completeness",
        f"+{clean_df.shape[1] - raw_cols} canonical derived features",
        "Converted to YYYY-MM-DD datetime objects",
        "billing_outlier_flag = 1 (Preserved for audit)"
    ]
})
print("Before vs. After Cleaning Reconciliation:")
print(comparison_table.to_string(index=False))


Before vs. After Cleaning Reconciliation:
       Dimension / Attribute  Before (Raw Data)           After (Clean Data)                    Delta / Transformation Action
                  Total Rows             55,500                       54,966                -534 exact duplicate rows removed
        Exact Duplicate Rows                534              0 (All Dropped) Purged identical repeats (First occurrence kept)
  Direct Identifiers Present Yes (Name, Doctor) No (Pseudonymized & Dropped)       patient_key (SHA-256) & doctor_id (DR-xxx)
Total Missing Values (Nulls)  0 (100% Complete)            0 (100% Complete)                     Maintained full completeness
       Total Feature Columns                 15                           32                   +17 canonical derived features
             Date Data Types    Object (String)               Datetime64[ns]         Converted to YYYY-MM-DD datetime objects
     Negative Billed Records                108                106 (Flagged)

## 5. Validation Assertions Execution
Validating the 15 data integrity, domain, and privacy assertions.

In [6]:
from src.validate import validate_clean_dataset

validation_report_df = validate_clean_dataset(
    clean_df_path='data/processed/hospital_clean.csv' if os.path.exists('data/processed/hospital_clean.csv') else '../data/processed/hospital_clean.csv',
    raw_df_path=raw_path
)


Validation report saved to outputs\tables\validation_report.csv
Mirrored validation report to phase2_preprocessing\validation_report.csv

DATA PREPROCESSING VALIDATION REPORT SUMMARY
               assertion_name                                      condition_checked  records_tested  violations_count status                                  details
       no_nulls_in_key_fields                         key_fields.isnull().sum() == 0           54966                 0   PASS           All records satisfy assertion.
          chronological_order                       discharge_date >= admission_date           54966                 0   PASS           All records satisfy assertion.
             non_negative_los                                          los_days >= 0           54966                 0   PASS           All records satisfy assertion.
      admission_id_uniqueness                      admission_id.nunique() == len(df)           54966                 0   PASS           All records s

## 6. Privacy-Safe Clean Dataset Sample Preview
Displaying sample records of the final analysis-ready dataset. Notice that no personal names or physician names exist anywhere.

In [7]:
# Display sample preview of final canonical dataset (excluding hospital to avoid synthetic name collisions)
preview_cols = [
    'admission_id', 'patient_key', 'age', 'age_group', 'gender', 
    'medical_condition', 'admission_date', 'discharge_date', 'los_days', 
    'doctor_id', 'insurance_provider', 'billing_amount', 'cost_per_day'
]
print(clean_df[preview_cols].head(5).to_string(index=False))


admission_id  patient_key  age age_group gender medical_condition admission_date discharge_date  los_days doctor_id insurance_provider  billing_amount  cost_per_day
     A032838 00011c86202c   77       65+   Male      Hypertension     2020-02-17     2020-03-15        27   DR-3670              Aetna       42,939.70      1,590.36
     A011347 00011c86202c   65       65+ Female           Obesity     2024-01-28     2024-02-07        10  DR-19745              Cigna       25,552.58      2,555.26
     A016723 0001a2301e3c   73       65+ Female           Obesity     2023-12-10     2023-12-14         4  DR-10308         Blue Cross       47,361.13     11,840.28
     A027341 0004280a8e1c   60     50-64   Male            Asthma     2020-09-11     2020-09-17         6  DR-32670   UnitedHealthcare       41,397.27      6,899.54
     A020004 0004280a8e1c   42     35-49 Female           Obesity     2022-06-27     2022-07-11        14  DR-12903   UnitedHealthcare       44,832.78      3,202.34
